# Part 3: Agent-driven policy research

This notebook scores seeds, tests your evaluator, and analyzes campaigns; the campaigns themselves run from the command line. The [Part 3 handout](../docs/part3_handout.md) has the rules, the metric, and the questions.

Every policy here runs on two replicas. Use the **CS 378 Assignment 4** kernel ([setup](../README.md#setup)).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())          # every path below is relative to the repository root
plt.rcParams["figure.dpi"] = 100

from agentkv import metric, score
from agentkv.trace import load

DEV = sorted(Path("data/traces/public_dev").glob("*.json"))
print(f"{len(DEV)} dev traces; full credit at S = {metric.FULL_CREDIT_SPEEDUP:.3f}")

## 1. Scoring

`score()` returns the S defined in the [handout](../docs/part3_handout.md#scoring): a geometric mean of per-trace speedups over the reference, first within each family and then across the eight families. The reference scores 1.000, which earns 10% of the grade; S ≥ 1.099 with no family below 0.90 earns the full 30%. Your policy cannot see family labels; your research system can.

In [ ]:
reference = score("policies/reference.py", split="public_dev")
print(reference.summary())
viz.families(reference)
plt.show()

## 2. The eight workload families

![Workload families: call structure and which parts of a prompt later calls can reuse.](../docs/figures/families.svg)

Calls per job, mean prompt and output lengths, the share of prompt tokens in previously seen blocks, and KV capacity per replica, for each `public_dev` trace:

In [ ]:
def summarize(path):
    t = load(path)
    calls = [n for _, n in t.llm_nodes()]
    seen, shared, total = set(), 0, 0
    for job in sorted(t.jobs, key=lambda j: j.arrival):
        for n in (x for x in job.nodes if x.type == "llm"):
            full = n.hashes[: n.n_prompt // t.memory.block_size]
            shared += sum(h in seen for h in full) * t.memory.block_size
            total += n.n_prompt
            seen.update(full)
    return {"trace": t.trace_id, "calls/job": len(calls) / len(t.jobs),
            "prompt": sum(n.n_prompt for n in calls) / len(calls),
            "output": sum(n.n_output for n in calls) / len(calls),
            "shared": shared / total, "kv tokens": t.memory.num_blocks * t.memory.block_size,
            "description": t.description}

rows = [summarize(p) for p in DEV]
print(f"{'trace':10} {'calls/job':>9} {'prompt':>8} {'output':>8} {'shared':>7} {'kv tokens':>10}")
for r in rows:
    print(f"{r['trace']:10} {r['calls/job']:9.1f} {r['prompt']:8.0f} {r['output']:8.0f} "
          f"{r['shared']:7.0%} {r['kv tokens']:10,d}")
print()
for r in rows[::4]:
    print(f"{r['trace']}: {r['description']}")

`viz.record(..., replicas=2)` shows where one trace's time goes: waiting for a slot, waiting for admission, or running, with memory below. Use it for Q5 and Q6.

In [ ]:
rec = viz.record("data/traces/public_dev/dev-W4-0.json", "policies/prefix_policy.py", replicas=2)
fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [2.4, 1]},
                         layout="constrained")
viz.timeline(rec, axes[0])
viz.memory(rec, axes[1])
plt.show()

## 3. Seeds and the evaluator

Score your candidate seeds first; Q2 measures your agents against the best one. `split="screen"` costs a quarter of `public_dev`.

In [ ]:
seeds = {"reference": ("policies/reference.py", None), "FIFO": ("policies/fifo.py", None),
         "prefix policy": ("policies/prefix_policy.py", None)}
for name, (path, params) in seeds.items():
    quick = score(path, split="screen", params=params)
    dev = score(path, split="public_dev", params=params)
    print(f"{name:14} screen S = {quick.speedup:.3f}   public_dev S = {dev.speedup:.3f}   "
          f"credit {dev.points:.1f}%")

Your evaluator decides what your agents optimize and see. The example returns the score, per-trace `diagnostics`, and an `objective` that is `None` for invalid candidates.

In [ ]:
import runpy

evaluator = runpy.run_path("examples/evaluator.py")      # research/evaluator.py once you have adapted it
feedback = evaluator["evaluate"]("policies/prefix_policy.py", split="screen")
print("objective:", feedback["objective"])
print(feedback["diagnostics"][:1500])

## 4. Your research system

`examples/research_system.py` is a minimal greedy loop. Decide, and report in Q1:

- what feedback agents see (scores, per-trace statistics, earlier hypotheses);
- how failed candidates (syntax, sandbox rules, the 50 ms or 4 MiB limit) are repaired;
- how many lineages run, and how the next parent is chosen;
- which split screens candidates before `public_dev`.

The offline run below writes the example's campaign files; `campaign_plots` reads its `log.jsonl`.

In [ ]:
from datetime import datetime

USE_BEDROCK = False
EVALUATOR = Path("examples/evaluator.py")             # research/evaluator.py once you have adapted it
DEMO_RUN = Path("runs") / f"example-demo-{datetime.now():%Y%m%d-%H%M%S-%f}"
OFFLINE = "" if USE_BEDROCK else "--offline"
!{sys.executable} examples/research_system.py --evaluator {EVALUATOR} --iters 3 {OFFLINE} --out {DEMO_RUN}

`campaign_plots` draws the best `public_dev` S so far against candidates scored and tokens spent, and marks finalists' `public_val` S. Use it for Q4.

In [ ]:
from matplotlib.ticker import MaxNLocator


def campaign_plots(log_path, start_S=1.0, finalists=None):
    rows = [json.loads(line) for line in Path(log_path).read_text().splitlines() if line.strip()]
    rows = [r for r in rows if not r.get("finalist")]
    best, tokens = start_S, 0
    xs, ts, bests, devs = [0], [0], [start_S], []
    for i, r in enumerate(rows, 1):
        tokens += r.get("tokens", 0)
        if r.get("valid") and r.get("dev_S"):
            best = max(best, r["dev_S"])
            devs.append((i, tokens, r["dev_S"]))
        xs.append(i); ts.append(tokens); bests.append(best)
    valid = sum(bool(r.get("valid")) for r in rows) / max(len(rows), 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True, layout="constrained")
    for ax, x, key in ((axes[0], xs, 0), (axes[1], ts, 1)):
        ax.step(x, bests, where="post", color=viz.COLORS["running"], label="best so far")
        ax.scatter([d[key] for d in devs], [d[2] for d in devs], s=12, color="#999999", label="candidate")
        for name, (cand, val_S) in (finalists or {}).items():
            point = next((d for d in devs if rows[d[0] - 1]["id"] == cand), None)
            if point:
                ax.scatter(point[key], val_S, marker="*", s=90, color=viz.COLORS["critical"], zorder=3)
                ax.annotate(f"{name} (val)", (point[key], val_S), fontsize=8, xytext=(3, 3),
                            textcoords="offset points")
        ax.axhline(1.0, color="gray", lw=0.8)
    axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))
    if tokens == 0:
        axes[1].text(0.5, 0.5, "no LLM tokens (offline run)", transform=axes[1].transAxes, ha="center")
    axes[0].set(xlabel="candidates scored", ylabel="public_dev S")
    axes[1].set(xlabel="LLM tokens")
    axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle(f"{log_path}: {len(rows)} candidates, {valid:.0%} valid")
    plt.show()


campaign_plots(DEMO_RUN / "log.jsonl", start_S=reference.speedup)    # then your own campaigns

## 5. Finalists

Score finalists on `public_val` as well as `public_dev`, and report the spread between the two.

In [ ]:
finalists = {
    # "p17": "runs/<campaign>/p17.py",
}
results = {name: (score(path, split="public_dev"), score(path, split="public_val"))
           for name, path in finalists.items()}
if results:
    fig, ax = plt.subplots(figsize=(5, 4.5))
    for name, (dev, val) in results.items():
        print(f"{name}: dev S = {dev.speedup:.4f}, val S = {val.speedup:.4f}, "
              f"valid on both = {dev.valid and val.valid}")
        ax.scatter(dev.speedup, val.speedup)
        ax.annotate(name, (dev.speedup, val.speedup), fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.axline((1, 1), slope=1, color="gray", ls="--", lw=0.8)
    ax.set(xlabel="public_dev S", ylabel="public_val S")
    plt.show()

## 6. The selected policy

Write the selected policy as pseudocode, then disable one mechanism at a time (in a copy or through a parameter) and score both public splits. A mechanism matters only if removing it changes S. Use the results for Q5.

In [ ]:
selected = "runs/<campaign>/<id>.py"
ablations = {
    # "no cache-aware routing": ("candidates/<id>_no_routing.py", None),
}
if Path(selected).exists():
    full = {split: score(selected, split=split) for split in ("public_dev", "public_val")}
    for name, (path, params) in ablations.items():
        for split in full:
            r = score(path, split=split, params=params)
            change = " ".join(f"{f} {r.by_family[f] / full[split].by_family[f]:.3f}" for f in r.by_family)
            print(f"{name:24} {split:10} S = {r.speedup:.4f} (full {full[split].speedup:.4f})  {change}")

## Questions

Answer Q1–Q6 from the [handout](../docs/part3_handout.md#research-report) in `assignment4_report.pdf`. The [README](../README.md#submission) lists what to submit.